# AXI test 27: the bigger test. Gemini 3.8 Flash on eight ARC-AGI-3 games, 60 moves each: the plain model (K), the model with the AXI ledger read-out (C), and the model with the read-out plus the notes (D)
**What this is.** The same model on eight public ARC-AGI-3 games (the four of Tests 24 to 26 and four more, chosen by a fixed rule written on the page), 60 moves each, three arms, one run per arm. **K** is the plain model with a kept chat. **C** is the model with the framework's Record, check, object view, untried list, conductor and the ledger read-out (arm C of Test 26). **D** is arm C plus one section, the Notes (the goal-readers of Update 106): which actions changed the board without moving a piece, what is new on this level, what earlier levels of the game suggest, and guesses (marked as guesses) at what completes a level. The structure behind the ledger is Rajnish Choubey's RFC-001 (the reading in game terms is for Sir to confirm).

The predictions and the reading rules were written before any call in `docs/GEMINI_ARC_BIG_TEST.md` (hash-locked). **This is not a measure of how well the model solves new games and not a leaderboard entry.**

**This test costs real money, and this notebook never turns billing on. It also takes hours: about 2 to 3.5 hours of calls per arm, and the whole test is meant to cost about 20 dollars.** The three arms are independent, so open this notebook **three times** (three Colab tabs), set `ARM` to `"K"`, `"C"` and `"D"` in Cell 5 of each, and run them side by side. Each saves every move to your Drive; if one stops, open it again, run Cells 2 to 5 and run its Cell 7 again: it carries on.

Steps: (1) type `ARM` and the prices in Cell 5; (2) run the pilot in Cell 6 once, in any one of the tabs (36 moves in all, well under a dollar), and send me its printout; (3) only then type a budget in dollars (`BUDGET_USD`, for that tab) and run Cell 7; (4) when all three arms are finished, run Cell 8 in any tab: it reads the three files, replays every move and prints the 18 lines.

**Keep each tab open** while its run is going.

### Cell 1: your key (once; no code to run)
The Colab Secret `GOOGLE_API_KEY` with Notebook access on (you did this for the earlier notebooks; it is the same key). The ARC toolkit needs no key of its own: it takes an anonymous one by itself.

### Cell 2: get the code from GitHub (the update with test 27 must be pushed to GitHub first)

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/gemini_arc_d.py"), "the code is not on GitHub yet: the update with test 27 must be pushed first"
print("code ok")

### Cell 3: install the Gemini library and the ARC toolkit (about 1 minute)

In [ ]:
!pip -q install -U google-genai arc-agi arcengine 2>&1 | tail -2

### Cell 4: connect Google Drive (a pop-up asks permission: Connect / Allow). Your progress and every reply are kept there

In [ ]:
import sys
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini_big")

### Cell 5: settings and the games. Fill the lines at the top (ARM first). PAID is True only after you turned billing on. The budget stays empty until you have read the pilot

In [ ]:
import json, time, logging, os
from google.colab import userdata, files
from axi.experiments import gemini_arc as G
from axi.experiments import gemini_arc_b as B
from axi.experiments import gemini_arc_c as C
from axi.experiments import gemini_arc_d as D
from axi.experiments.api_llm import QuotaStop, ServiceBusy, BudgetStop

ARM = None           # which arm THIS tab runs: "K" (plain model), "C" (ledger read-out) or "D" (read-out plus notes). One tab per arm
PAID = None          # True ONLY after you turned billing on for this key in Google AI Studio
PRICE_IN = None      # dollars per 1,000,000 INPUT tokens of Gemini 3.8 Flash
PRICE_OUT = None     # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output)
PRICE_CACHED = None  # dollars per 1,000,000 CACHED input tokens (type it: without it cached tokens are counted at the full price)
BUDGET_USD = None    # the most you allow THIS tab's arm to cost. Leave None until you have read the pilot in Cell 6
THINKING = None      # None = the model's own default

assert ARM in ("K", "C", "D"), "Set ARM to \"K\", \"C\" or \"D\" (one arm per tab)."
assert PAID in (True, False), "Set PAID: True if you turned billing on, False for a free key."
try: KEY = userdata.get("GOOGLE_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: a Secret named GOOGLE_API_KEY with Notebook access on.")
if PAID: assert PRICE_IN and PRICE_OUT, "Type PRICE_IN and PRICE_OUT (dollars per million tokens). The notebook has no built-in price on purpose."
TAG = str(THINKING or "default"); CACHE = "axi_gemini_big_cache_%s_%s.jsonl" % (ARM, TAG)           # one reply file per arm, so that tabs running side by side never overwrite each other
print("reply file:", store.restore(CACHE))
guard = dict(budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED) if (PAID and BUDGET_USD) else dict(price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED)
llm, MODEL = G.GL.build_chat_llm(KEY, store.path(CACHE), daily_limit=None if PAID else 20, min_interval=0.3 if PAID else 4.5, on_write=lambda: store.persist(CACHE), thinking=THINKING, **guard)
META = dict(model=MODEL, thinking=THINKING)
sp = llm.spent_usd()
print("arm:", ARM, "| paid:", PAID, "| model:", MODEL, "| thinking:", THINKING or "model default", "| budget:", ("$%.2f" % BUDGET_USD) if BUDGET_USD else "NOT SET (pilot only)", "| spent so far with these settings:", ("$%.2f" % sp) if sp is not None else "-")
import arc_agi
logging.disable(logging.INFO)
ARC = arc_agi.Arcade()
def make_env(game):
    env = ARC.make(game, seed=G.GAME_SEED)
    assert env is not None, "could not make " + game
    return env
OFFERED = [str(getattr(e, "game_id", e)) for e in ARC.get_environments()]
GAMES = D.pick_games(OFFERED)                                                                    # the rule on the page: the four of the earlier tests, then the others by the sha256 of their id
GF = "arc_big_games.json"; print("games file:", store.restore(GF))
if os.path.exists(store.path(GF)):
    saved = json.load(open(store.path(GF)))
    if saved != GAMES: print("NOTE: the toolkit's list differs from the list saved on your Drive; the saved list is used, so that every arm plays the same games.")
    GAMES = saved
else:
    json.dump(GAMES, open(store.path(GF), "w")); store.persist(GF)
for g in GAMES: assert g in OFFERED, g + " is not in the list of games the toolkit offers today"
assert D.PILOT_GAME in GAMES, "the pilot game is not among the games"
print(len(GAMES), "games:", ", ".join(GAMES))
RESET = "The free quota resets at midnight Pacific time. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do not run it again and again."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        if isinstance(e, BudgetStop): pass
        elif isinstance(e, ServiceBusy): print(BUSY)
        else: print(RESET)
        return None
def need_budget(): assert PAID and BUDGET_USD, "Read the pilot (Cell 6), then type BUDGET_USD in Cell 5, run Cell 5 again, and run this cell again."
def save(res, name):
    p = store.path(name + ".json"); json.dump(res, open(p, "w")); store.persist(name + ".json"); return p
def persisting(name, inner):
    print("progress file:", store.restore(name + ".jsonl"))
    def cb(*a):
        inner(*a)
        if a[0] % 10 == 0 or a[0] == a[1]: store.persist(name + ".jsonl")
    return cb
def prog(t, n, r):
    if t % 5 == 0 or t == n:
        sp = llm.spent_usd(); print(f"move {t}/{n} (game {r['game']} move {r['m']}) | prompt {r['prompt_tokens']:,} (cached {r['cache_tokens']:,}) | {r['seconds']:.1f} s | {r['kind']} legal {r['reply_ok']} changed {r['changed']} | levels {r['levels']}/{r['win_levels']} | spent {('$%.2f' % sp) if sp is not None else '-'}")

### Cell 6: the PILOT (run it once, in any one tab). 12 real moves of each arm in the first game (well under a dollar). It prints how fast the plain model's chat grows, how many moves the framework played itself, how many ledger lines and note lines the model was shown, the speed, and the projected cost of each arm and of the three together at your prices

In [ ]:
assert PAID, "The pilot needs a paid key (about 36 calls)."
pk = go(lambda: G.run(llm, make_env, "K", store.path("arc_big_pilot_K.jsonl"), games=(D.PILOT_GAME,), moves=D.PILOT_MOVES, meta=META))
pc = go(lambda: C.run(llm, make_env, store.path("arc_big_pilot_C.jsonl"), games=(D.PILOT_GAME,), moves=D.PILOT_MOVES, meta=META))
pd = go(lambda: D.run(llm, make_env, store.path("arc_big_pilot_D.jsonl"), games=(D.PILOT_GAME,), moves=D.PILOT_MOVES, meta=META))
if pk and pc and pd:
    for n in ("K", "C", "D"): store.persist("arc_big_pilot_%s.jsonl" % n)
    D.print_stats("pilot K (plain model)", pk["rows"]); D.print_stats("pilot C (ledger read-out)", pc["rows"]); D.print_stats("pilot D (read-out and notes)", pd["rows"])
    rep = D.pilot_report(pk["rows"], pc["rows"], pd["rows"], PRICE_IN, PRICE_OUT, PRICE_CACHED)
    print()
    for k, v in rep.items(): print(f"  {k}: {v if not isinstance(v, float) else round(v, 3)}")
    print("\nRead these before the budget: k_growth_per_move and k_total_prompt_tokens are the plain model's kept chat (they grow with the square of the moves); cost_k, cost_c and cost_d are the three arms at your prices over the whole run; c_forced_share and d_forced_share must stay below 0.4 over the whole run.")
    print("Then type BUDGET_USD in Cell 5 (a budget for this tab's arm) and send me this printout before you run Cell 7.")

### Cell 7: THE ARM OF THIS TAB (the one you set in Cell 5). 8 games x 60 moves, about 2 to 3.5 hours. A line every 5 moves. If it stops, run Cells 2 to 5 and this cell again

In [ ]:
need_budget()
name = "arc_big_%s_%s" % (ARM, TAG)
RUN = dict(
    K=lambda: G.run(llm, make_env, "K", store.path(name + ".jsonl"), games=GAMES, moves=D.BIG_MOVES, meta=META, max_tokens=D.K_TOKEN_CAP, progress=persisting(name, prog)),
    C=lambda: C.run(llm, make_env, store.path(name + ".jsonl"), games=GAMES, moves=D.BIG_MOVES, meta=META, max_tokens=D.C_TOKEN_CAP, progress=persisting(name, prog)),
    D=lambda: D.run(llm, make_env, store.path(name + ".jsonl"), games=GAMES, moves=D.BIG_MOVES, meta=META, max_tokens=D.D_TOKEN_CAP, progress=persisting(name, prog)))
res = go(RUN[ARM])
if res:
    D.print_stats("ARM " + ARM, res["rows"]); D.print_games(res["rows"]); store.persist(name + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(res, name))

### Cell 8: the READING (run it once all three arms are finished, in any tab). Reads the three files from your Drive (no call to the model), replays every saved move of every arm into fresh copies of the games (arms C and D also rebuild the ledgers, and D the clears), prints the tables and the 18 lines with yes / NO, and saves the report. Send me the output and the downloaded file

In [ ]:
arms = {}
for a in ("K", "C", "D"):
    nm = "arc_big_%s_%s" % (a, TAG); print(nm, store.restore(nm + ".jsonl"))
    try: arms[a] = D.load_arm(store.path(nm + ".jsonl"))
    except Exception as e: print("arm", a, "not read:", e); arms[a] = None
ok = {}
for a, fn in (("K", lambda r: G.replay(r, make_env, "K", GAMES, D.BIG_MOVES)), ("C", lambda r: C.replay(r, make_env, GAMES, D.BIG_MOVES)), ("D", lambda r: D.replay(r, make_env, GAMES, D.BIG_MOVES))):
    if not arms[a]: continue
    try: fn(arms[a]["rows"]); ok[a] = True
    except Exception as e: print("replay", a + ":", e); ok[a] = False
checks = D.verdict(arms["K"], arms["C"], arms["D"], ok, games=GAMES, moves=D.BIG_MOVES)
for a, t in (("K", "K (plain model)"), ("C", "C (ledger read-out)"), ("D", "D (read-out and notes)")):
    if arms[a]: print(); D.print_stats("ARM " + t, arms[a]["rows"]); D.print_games(arms[a]["rows"])
print("\nlevels completed in each game:"); D.print_levels(GAMES, K=arms["K"], C=arms["C"], D=arms["D"])
D.print_verdict(checks)
print("usage:", llm.usage(), "| spent: $%.2f" % llm.spent_usd())
files.download(save(dict(K=arms["K"], C=arms["C"], D=arms["D"], games=GAMES, checks=checks, replays=ok), "gemini_arc_big_all_%s" % TAG))

### Done
Files downloaded: one file per arm and `gemini_arc_big_all_*.json`. Upload `gemini_arc_big_all_default.json` to the chat. Nothing here says anything about any company's own service: it measures this model on these games.